# Phase 1.2: Raw Data Inspection (MODIS vs VIIRS)

This notebook inspects and compares the raw CSV data downloaded from the NASA FIRMS API for **MODIS** (Terra & Aqua) and **VIIRS** (Suomi-NPP, NOAA-20, NOAA-21).

We compare:
1. Schema and data types (head() and dtypes)
2. Column differences between sensors
3. Unique values of key operational fields: confidence, daynight, and satellite
4. Sensor characteristics and calibration considerations


In [1]:
import glob
from pathlib import Path
import pandas as pd

# Paths relative to DATA directory
raw_dir = Path('../data/raw')
modis_files = sorted(list((raw_dir / 'palisades_la' / 'MODIS').glob('*.csv')))
viirs_files = sorted(list((raw_dir / 'palisades_la' / 'VIIRS').glob('*.csv')))

# Select non-empty CSVs for inspection
modis_file = None
for f in modis_files:
    if f.stat().st_size > 300:
        modis_file = f
        break

viirs_file = None
for f in viirs_files:
    if f.stat().st_size > 300:
        viirs_file = f
        break

print(f'Selected MODIS file: {modis_file}')
print(f'Selected VIIRS file: {viirs_file}')

df_modis = pd.read_csv(modis_file) if modis_file else pd.DataFrame()
df_viirs = pd.read_csv(viirs_file) if viirs_file else pd.DataFrame()

print(f'MODIS detections loaded: {len(df_modis):,}')
print(f'VIIRS detections loaded: {len(df_viirs):,}')


Selected MODIS file: ..\data\raw\palisades_la\MODIS\2005.csv
Selected VIIRS file: ..\data\raw\palisades_la\VIIRS\2012.csv
MODIS detections loaded: 28
VIIRS detections loaded: 13


In [2]:
print('=== MODIS HEAD (first 3 rows) ===')
print(df_modis.head(3))
print('\n=== MODIS DATA TYPES ===')
print(df_modis.dtypes)

print('\n========================================================')
print('=== VIIRS HEAD (first 3 rows) ===')
print(df_viirs.head(3))
print('\n=== VIIRS DATA TYPES ===')
print(df_viirs.dtypes)


=== MODIS HEAD (first 3 rows) ===
   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0   34.1924  -118.7123       305.0   2.9    1.6  2005-09-29       910   
1   34.1938  -118.7358       353.9   3.0    1.6  2005-09-29       910   
2   34.1946  -118.7437       360.0   3.0    1.6  2005-09-29       910   

  satellite instrument  confidence  version  bright_t31    frp daynight  type  \
0      Aqua      MODIS           5     6.03       295.0   27.9        N     0   
1      Aqua      MODIS         100     6.03       299.2  343.0        N     0   
2      Aqua      MODIS         100     6.03       300.2  417.8        N     0   

  processing  
0         SP  
1         SP  
2         SP  

=== MODIS DATA TYPES ===
latitude      float64
longitude     float64
brightness    float64
scan          float64
track         float64
acq_date          str
acq_time        int64
satellite         str
instrument        str
confidence      int64
version       float64
bright_t31    float6

In [3]:
modis_cols = set(df_modis.columns)
viirs_cols = set(df_viirs.columns)

print('=== COLUMN COMPARISON ===')
print('Columns unique to MODIS (not in VIIRS):', sorted(list(modis_cols - viirs_cols)))
print('Columns unique to VIIRS (not in MODIS):', sorted(list(viirs_cols - modis_cols)))
print('Columns common to both sensors:', sorted(list(modis_cols & viirs_cols)))


=== COLUMN COMPARISON ===
Columns unique to MODIS (not in VIIRS): ['bright_t31', 'brightness']
Columns unique to VIIRS (not in MODIS): ['bright_ti4', 'bright_ti5']
Columns common to both sensors: ['acq_date', 'acq_time', 'confidence', 'daynight', 'frp', 'instrument', 'latitude', 'longitude', 'processing', 'satellite', 'scan', 'track', 'type', 'version']


In [4]:
print('=== UNIQUE VALUES: CONFIDENCE ===')
print('MODIS Confidence (Numeric):', sorted(df_modis['confidence'].dropna().unique().tolist()) if 'confidence' in df_modis else 'N/A')
print('VIIRS Confidence (Categorical):', sorted(df_viirs['confidence'].dropna().unique().tolist()) if 'confidence' in df_viirs else 'N/A')

print('\n=== UNIQUE VALUES: SATELLITE ===')
print('MODIS Satellites:', df_modis['satellite'].dropna().unique().tolist() if 'satellite' in df_modis else 'N/A')
print('VIIRS Satellites:', df_viirs['satellite'].dropna().unique().tolist() if 'satellite' in df_viirs else 'N/A')

print('\n=== UNIQUE VALUES: DAYNIGHT ===')
print('MODIS Day/Night:', df_modis['daynight'].dropna().unique().tolist() if 'daynight' in df_modis else 'N/A')
print('VIIRS Day/Night:', df_viirs['daynight'].dropna().unique().tolist() if 'daynight' in df_viirs else 'N/A')


=== UNIQUE VALUES: CONFIDENCE ===
MODIS Confidence (Numeric): [5, 38, 46, 55, 58, 67, 80, 84, 86, 89, 90, 91, 92, 94, 98, 99, 100]
VIIRS Confidence (Categorical): ['l', 'n']

=== UNIQUE VALUES: SATELLITE ===
MODIS Satellites: ['Aqua', 'Terra']
VIIRS Satellites: ['N']

=== UNIQUE VALUES: DAYNIGHT ===
MODIS Day/Night: ['N', 'D']
VIIRS Day/Night: ['N', 'D']


### Key Observations and Differences

1. **Confidence Representation**:
   - **MODIS**: Encoded as a **continuous numeric integer from 0 to 100**, representing detection confidence percentage calculated by the standard contextual fire algorithm. In our cleaning rules, we retain $\ge 30$, where 30–79 is mapped to *nominal* and 80–100 to *high*.
   - **VIIRS**: Encoded as a **one-letter categorical code**: 'l' (low), 'n' (nominal), or 'h' (high). The I-band active fire algorithm does not output a continuous 0–100 scale because of the sub-pixel saturation and multi-threshold criteria used at 375 m resolution. Per our cleaning rules, 'l' detections are dropped and preserved in a separate audit file.

2. **Spectral Brightness Temperature Bands**:
   - **MODIS**: Uses rightness (4 µm, channel 21/22) and right_t31 (11 µm, channel 31).
   - **VIIRS**: Uses right_ti4 (3.75 µm, I-band 4) and right_ti5 (11.45 µm, I-band 5).

3. **Spatial Resolution**:
   - **MODIS**: Nominal spatial resolution of **1 km** at nadir.
   - **VIIRS**: Nominal spatial resolution of **375 m** at nadir, capable of detecting fires with substantially lower radiative output and smaller physical footprint.

4. **Satellite Identifiers**:
   - **MODIS**: 'Terra' (morning pass) and 'Aqua' (afternoon pass).
   - **VIIRS**: 'N' (Suomi-NPP), 'N20' (NOAA-20 / JPSS-1), and 'N21' (NOAA-21 / JPSS-2).
